# Assignment 8 — Regularized Regression for Grade Prediction (Ridge / Lasso / ElasticNet)
**Dataset:** UCI Student Performance (`student-por.csv`)  
**Target:** continuous `G3`; inputs exclude G1/G2/G3

**Aim:** Compare OLS vs Ridge/Lasso/ElasticNet with CV-tuned alpha; report train/test scores, MAE/RMSE/R², coefficient shrinkage, Lasso sparsity, and explain overfitting vs regularization.


## 1. Library imports

In [1]:
import io, os
import urllib.request, zipfile
import numpy as np, pandas as pd
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression, RidgeCV, LassoCV, ElasticNetCV
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
sns.set_theme(style="whitegrid")
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
os.makedirs("outputs", exist_ok=True)
print("random_state =", RANDOM_STATE)

random_state = 42


## 2. Dataset loading (official UCI source)

In [2]:
UCI_URL = "https://archive.ics.uci.edu/ml/machine-learning-databases/00320/student.zip"
raw = urllib.request.urlopen(UCI_URL, timeout=60).read()
zf = zipfile.ZipFile(io.BytesIO(raw))
with zf.open("student-por.csv") as f:
    df = pd.read_csv(f, sep=";")
print("Shape:", df.shape)

Shape: (649, 33)


## 3. Dataset preview

In [3]:
df.head()

,school,sex,age,address,famsize,Pstatus,Medu,Fedu,Mjob,Fjob,...,famrel,freetime,goout,Dalc,Walc,health,absences,G1,G2,G3
0,GP,F,18,U,GT3,A,4,4,at_home,teacher,...,4,3,4,1,1,3,4,0,11,11
1,GP,F,17,U,GT3,T,1,1,at_home,other,...,5,3,3,1,1,3,2,9,11,11
2,GP,F,15,U,LE3,T,1,1,at_home,other,...,4,3,2,2,3,3,6,12,13,12
3,GP,F,15,U,GT3,T,4,2,health,services,...,3,2,2,1,1,5,0,14,14,14
4,GP,F,16,U,GT3,T,3,3,other,other,...,4,3,2,1,2,5,0,11,13,13


In [4]:
print(df["G3"].describe()); print("Missing:", int(df.isnull().sum().sum()))

count    649.000000
mean      11.906009
std        3.230656
min        0.000000
25%       10.000000
50%       12.000000
75%       14.000000
max       19.000000
Name: G3, dtype: float64
Missing: 0


## 4. Data preprocessing — pipeline inputs

In [5]:
X_raw = df.drop(columns=["G1", "G2", "G3"])
y = df["G3"].to_numpy()
cat_cols = X_raw.select_dtypes(include=["object"]).columns.tolist()
num_cols = X_raw.select_dtypes(exclude=["object"]).columns.tolist()
preprocess = ColumnTransformer([
    ("num", StandardScaler(), num_cols),
    ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), cat_cols)])
X_train_raw, X_test_raw, y_train, y_test = train_test_split(
    X_raw, y, test_size=0.2, random_state=RANDOM_STATE)
print(f"Train {X_train_raw.shape} Test {X_test_raw.shape}")
Xtr = preprocess.fit_transform(X_train_raw)
Xte = preprocess.transform(X_test_raw)
feat_names = list(preprocess.get_feature_names_out())
print("Encoded dim:", Xtr.shape[1])

Train (519, 30) Test (130, 30)


Encoded dim: 56


C:\Users\ns479\AppData\Local\Temp\ipykernel_12960\4220838251.py:3: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  cat_cols = X_raw.select_dtypes(include=["object"]).columns.tolist()


## 5. Model implementation — OLS vs RidgeCV / LassoCV / ElasticNetCV (alpha via CV)

In [6]:
alphas = np.logspace(-3, 2, 30)
ols = LinearRegression().fit(Xtr, y_train)
ridge = RidgeCV(alphas=alphas, cv=5).fit(Xtr, y_train)
lasso = LassoCV(alphas=alphas, cv=5, random_state=RANDOM_STATE, max_iter=10000).fit(Xtr, y_train)
enet = ElasticNetCV(alphas=alphas, l1_ratio=[0.2, 0.5, 0.8], cv=5,
                    random_state=RANDOM_STATE, max_iter=10000).fit(Xtr, y_train)
print(f"Ridge best alpha: {ridge.alpha_}")
print(f"Lasso best alpha: {lasso.alpha_}")
print(f"ElasticNet best alpha: {enet.alpha_}, l1_ratio: {enet.l1_ratio_}")

Ridge best alpha: 45.2035365636025
Lasso best alpha: 0.02395026619987486
ElasticNet best alpha: 0.0529831690628371, l1_ratio: 0.2


## 6. Evaluation — train/test scores + MAE/RMSE/R²

In [7]:
rows = []
for name, m in [("OLS-LinearRegression", ols), ("Ridge", ridge), ("Lasso", lasso), ("ElasticNet", enet)]:
    ptr, pte = m.predict(Xtr), m.predict(Xte)
    rows.append({"model": name,
        "train_R2": r2_score(y_train, ptr), "test_R2": r2_score(y_test, pte),
        "test_MAE": mean_absolute_error(y_test, pte),
        "test_RMSE": float(np.sqrt(mean_squared_error(y_test, pte)))})
comp = pd.DataFrame(rows)
print(comp.round(4).to_string(index=False))
comp.round(4).to_csv("outputs/model_comparison.csv", index=False)

               model  train_R2  test_R2  test_MAE  test_RMSE
OLS-LinearRegression    0.3874   0.1602    2.1564     2.8618
               Ridge    0.3761   0.2013    2.0604     2.7908
               Lasso    0.3769   0.1949    2.0876     2.8020
          ElasticNet    0.3782   0.1980    2.0726     2.7966


## 7. Tables and graphs — coefficients, Lasso sparsity, comparison

In [8]:
coef_df = pd.DataFrame({"feature": feat_names, "OLS": ols.coef_, "Ridge": ridge.coef_,
    "Lasso": lasso.coef_, "ElasticNet": enet.coef_})
print(coef_df.reindex(coef_df["OLS"].abs().sort_values(ascending=False).index).head(15).round(4).to_string(index=False))
coef_df.to_csv("outputs/coefficients.csv", index=False)
n_zero_lasso = int((np.abs(lasso.coef_) < 1e-10).sum())
print(f"\nLasso zero coefficients: {n_zero_lasso} / {len(feat_names)}")
print(f"L1 norm OLS={np.abs(ols.coef_).sum():.3f} Ridge={np.abs(ridge.coef_).sum():.3f} "
      f"Lasso={np.abs(lasso.coef_).sum():.3f} EN={np.abs(enet.coef_).sum():.3f}")
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
melt = comp.melt(id_vars="model", value_vars=["test_MAE", "test_RMSE"])
sns.barplot(data=melt, x="variable", y="value", hue="model", ax=axes[0])
axes[0].set_title("Test error (lower is better)")
sns.barplot(data=comp, x="model", y="test_R2", ax=axes[1])
axes[1].set_title("Test R² (higher is better)"); axes[1].tick_params(axis="x", rotation=15)
plt.tight_layout(); plt.savefig("outputs/comparison.png", dpi=150); plt.close()
top = coef_df.reindex(coef_df["OLS"].abs().sort_values(ascending=False).index).head(12)
plt.figure(figsize=(11, 5.5))
x = np.arange(len(top)); w = 0.2
for i, c in enumerate(["OLS", "Ridge", "Lasso", "ElasticNet"]):
    plt.bar(x + (i-1.5)*w, top[c], width=w, label=c)
plt.xticks(x, top["feature"], rotation=45, ha="right"); plt.ylabel("Coefficient")
plt.title("Top-12 coefficient comparison (shrinkage toward zero)")
plt.legend(); plt.tight_layout(); plt.savefig("outputs/coefficients.png", dpi=150); plt.close()
print("Saved: outputs/comparison.png, outputs/coefficients.png")

               feature     OLS   Ridge   Lasso  ElasticNet
         num__failures -0.8841 -0.8272 -0.8598     -0.8466
       cat__higher_yes  0.8435  0.5673  0.0000      0.6332
        cat__higher_no -0.8435 -0.5673 -1.4140     -0.6332
    cat__schoolsup_yes -0.8017 -0.4995 -0.0000     -0.5684
     cat__schoolsup_no  0.8017  0.4995  1.3021      0.5684
        cat__school_GP  0.6968  0.5781  1.3537      0.6247
        cat__school_MS -0.6968 -0.5781 -0.0000     -0.6244
     cat__Fjob_at_home  0.6419  0.2204  0.0643      0.2021
      cat__Mjob_health  0.6017  0.2480  0.1761      0.2312
     cat__reason_other -0.5656 -0.3878 -0.5591     -0.4388
    cat__Fjob_services -0.4574 -0.3187 -0.4277     -0.3631
   cat__guardian_other  0.4561  0.1815  0.0532      0.1719
     cat__Mjob_at_home -0.4432 -0.2262 -0.1613     -0.2263
cat__reason_reputation  0.4368  0.3258  0.2968      0.3340
      cat__Fjob_health -0.3992 -0.0715 -0.0000     -0.0000

Lasso zero coefficients: 25 / 56
L1 norm OLS=15.274 Rid

Saved: outputs/comparison.png, outputs/coefficients.png


## 8. Result and conclusion — overfitting and regularization
- **Overfitting:** OLS minimizes training error only; with many one-hot columns it can learn large-magnitude coefficients that fit noise, giving a train/test R² gap.
- **Ridge (L2)** shrinks all coefficients smoothly (best alpha by 5-fold CV) — keeps all features, reduces variance.
- **Lasso (L1)** shrinks some coefficients *exactly to zero* (count printed above) — automatic feature selection.
- **ElasticNet** mixes L1+L2 (alpha + l1_ratio by CV) — robust when features are correlated.
- All alphas/scores/coefficients were computed at runtime (`random_state=42`); see `outputs/model_comparison.csv`.
